# L2 · Extracción con Docling y seccionado (en curso)

**Estado:** extracción real hecha; la cascada de seccionado y el criterio (≥ 95 % de artículos de la LA/FT) están pendientes. Este notebook muestra lo que ya funciona, con Docling real y sin simulaciones.

Configuración medida en la Pi limitada a 2 CPU (docs/14 #13): 2 hilos, sin OCR, tablas `fast`, tandas de 10 páginas, subproceso con timeout.

In [1]:
import os, sys, time
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

from backend.config import settings as s
from backend.ingest.docling_parser import extraer, limpiar_cache, ExtraccionError
from backend.sectioner import parsear_bloques

NORMAS = Path(os.environ.get("NORMAS_DIR", RAIZ.parent.parent / "comparador-normativas-ec-v1" / "Normativa2026"))
PDF = NORMAS / "L1-XVI-cap-III.pdf"   # norma corta de la Superintendencia de Bancos (3 páginas)
print("Docling:", f"tablas={s.DOCLING_TABLES}", f"hilos={s.DOCLING_THREADS}", f"timeout={s.DOCLING_TIMEOUT_S}s", f"tanda={s.DOCLING_CHUNK_PAGES} págs")

Docling: tablas=fast hilos=2 timeout=1500s tanda=10 págs


## 1 · Extracción real, con progreso

In [2]:
limpiar_cache()
avance = []
t0 = time.perf_counter()
bloques = extraer(PDF, lambda hechas, total: avance.append((hechas, total)))
t_extraccion = time.perf_counter() - t0

tipos = {}
for b in bloques:
    tipos[b["tipo"]] = tipos.get(b["tipo"], 0) + 1
print(f"{len(bloques)} bloques en {t_extraccion:.1f}s · progreso {avance}")
print("tipos:", tipos)
for b in bloques[:6]:
    print(f"  p.{b['pagina']} [{b['tipo']}] {b['texto'][:70]!r}")

43 bloques en 48.2s · progreso [(3, 3)]
tipos: {'text': 21, 'section_header': 3, 'list_item': 16, 'tabla': 3}
  p.1 [text] 'Codificación de las Normas de la Superintendencia de Bancos'
  p.1 [section_header] 'LIBRO I.- NORMAS DE CONTROL PARA LAS ENTIDADES DE LOS SECTORES FINANCI'
  p.1 [text] 'TÍTULO XVI.- DEL PROCESO DE FUSIÓN EXTRAORDINARIA, DE EXCLUSIÓN Y TRAN'
  p.1 [text] 'CAPÍTULO III.- NORMAS PARA LA DESIGNACIÓN DE LIQUIDADORES DE LAS ENTID'
  p.1 [section_header] 'SECCIÓN I.- DE LOS LIQUIDADORES EXTERNOS'
  p.1 [text] 'ARTÍCULO 1.- Para que una persona sea designada por el Superintendente'


## 2 · Caché por SHA-256: el mismo PDF no vuelve a pasar por Docling

In [3]:
t0 = time.perf_counter()
otra_vez = extraer(PDF)
t_cache = time.perf_counter() - t0
print(f"segunda llamada: {t_cache:.3f}s · mismo resultado: {otra_vez == bloques}")

segunda llamada: 0.017s · mismo resultado: True


## 3 · Los PDFs no válidos se rechazan antes de llegar a Docling

In [4]:
import tempfile
falso = Path(tempfile.mkdtemp()) / "falso.pdf"; falso.write_text("no es un pdf")
try:
    extraer(falso)
except ExtraccionError as e:
    print(e.codigo, "·", e.para_usuario()["mensaje"])

ERR-ING-003 · No pudimos abrir este archivo. Verifica que sea un PDF válido y sin contraseña.


## 4 · Seccionado con el nivel 1 (patrones) — la cascada completa está pendiente

In [5]:
secciones = parsear_bloques(bloques, doc_id="SB-III", tipo_doc="normativa")
arts = [x for x in secciones if x.nivel == "articulo"]
print(f"artículos detectados: {len(arts)}")
for a in arts[:3]:
    print(f"  {a.identificador} · ruta={a.ruta} · págs {a.pagina_inicio}-{a.pagina_fin} · {a.texto_literal[:50]!r}")

artículos detectados: 8
  ARTÍCULO 1 · ruta=['LIBRO I', 'TÍTULO XVI', 'CAPÍTULO III', 'SECCIÓN I'] · págs 1-1 · 'ARTÍCULO 1.- Para que una persona sea designada po'
  ARTÍCULO 2 · ruta=['LIBRO I', 'TÍTULO XVI', 'CAPÍTULO III', 'SECCIÓN I'] · págs 1-2 · 'ARTÍCULO 2.- No podrán ser designados como liquida'
  ARTÍCULO 3 · ruta=['LIBRO I', 'TÍTULO XVI', 'CAPÍTULO III', 'SECCIÓN I'] · págs 2-2 · 'ARTÍCULO 3.- El Superintendente de Bancos designar'


## 5 · Chequeos de lo implementado

In [6]:
checks = {
    "extracción real (hay bloques con texto)": len(bloques) > 0,
    "incluye tablas": tipos.get("tabla", 0) > 0,
    "reporta progreso hasta el total": bool(avance) and avance[-1][0] == avance[-1][1],
    "la caché devuelve lo mismo, sin Docling": otra_vez == bloques and t_cache < 1,
    "el PDF no válido se rechaza": True,
    "el seccionado detecta artículos": len(arts) > 0,
}
for k, v in checks.items():
    print(("✅" if v else "❌"), k)
assert all(checks.values())
print("\nPendiente de L2: cascada completa, criterio ≥ 95 % en la LA/FT, pruebas de qa-ia.")

✅ extracción real (hay bloques con texto)
✅ incluye tablas
✅ reporta progreso hasta el total
✅ la caché devuelve lo mismo, sin Docling
✅ el PDF no válido se rechaza
✅ el seccionado detecta artículos

Pendiente de L2: cascada completa, criterio ≥ 95 % en la LA/FT, pruebas de qa-ia.
